Install all the dependencies

In [ ]:
!pip install langchain chromadb faiss-cpu sentence-transformers pandas

Import all the installed dependencies

import pandas as pd
import numpy as np
from typing import List, Tuple

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.docstore.document import Document

Load Dataset

In [ ]:
url = "https://raw.githubusercontent.com/Bluedata-Consulting/GAAPB01-training-code-base/refs/heads/main/Assignments/assignment2dataset.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
df.head()

Prepare Course Documents

In [ ]:
# each course becomes a "document" with metadata
docs = []
for idx, row in df.iterrows():
    content = f"{row['title']} - {row['description']}"
    docs.append(Document(page_content=content, metadata={"course_id": row['course_id']}))
    
print("Prepared docs:", len(docs))

In [ ]:
# use HuggingFace model for semantic embeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# build FAISS index
vectorstore = FAISS.from_documents(docs, embedding_model)

print("Vector DB built with", len(docs), "courses.")

In [ ]:
# recommentation function for courses
def recommend_courses(profile: str, completed_ids: List[str], k: int = 5) -> List[Tuple[str, float]]:
    """
    Given a learner profile and completed course IDs, return top-k recommendations.
    """
    # Encode profile as embedding
    results = vectorstore.similarity_search_with_score(profile, k=20)
    
    # Filter out completed courses
    filtered = [(r.metadata["course_id"], score, r.page_content) 
                for r, score in results if r.metadata["course_id"] not in completed_ids]
    
    # Take top-k
    topk = sorted(filtered, key=lambda x: x[1])[:k]
    
    return [(cid, float(score)) for cid, score, _ in topk]

In [ ]:
# Sample Input Queries

test_queries = [
    ("I’ve completed the 'Python Programming for Data Science' course and enjoy data visualization. What should I take next?", ["C101"]),
    ("I know Azure basics and want to manage containers and build CI/CD pipelines. Recommend courses.", ["C202"]),
    ("My background is in ML fundamentals; I’d like to specialize in neural networks and production workflows.", ["C303"]),
    ("I want to learn to build and deploy microservices with Kubernetes—what courses fit best?", ["C404"]),
    ("I’m interested in blockchain and smart contracts but have no prior experience. Which courses do you suggest?", []),
]

Create Evaluation and collect results

In [ ]:
evaluation_results = []

for profile, completed in test_queries:
    recs = recommend_courses(profile, completed, k=5)
    for cid, score in recs:
        course_title = df[df["course_id"] == cid]["title"].values[0]
        evaluation_results.append({
            "profile": profile,
            "recommended_course_id": cid,
            "recommended_course_title": course_title,
            "similarity_score": score
        })

eval_df = pd.DataFrame(evaluation_results)
eval_df


Save evaluation results

In [ ]:
eval_df.to_csv("assignment2_recommendations.csv", index=False)
print("Saved results to assignment2_recommendations.csv")
